In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# Carregar variáveis de ambiente
load_dotenv()
chat = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key = os.getenv("ROUTER_API"),
    model="gpt-oss-120b:free", 
    temperature=0.7
)

In [2]:
# Cadeia de Química
prompt_quimica = ChatPromptTemplate.from_template(
    "Você é um especialista em química. Responda: {input}"
)
chain_quimica = prompt_quimica | chat

# Cadeia de Geografia
prompt_geo = ChatPromptTemplate.from_template(
    "Você é um especialista em geografia. Responda: {input}"
)
chain_geo = prompt_geo | chat


In [3]:
router_prompt = ChatPromptTemplate.from_template("""
Classifique a pergunta do usuário em uma das categorias:
- quimica
- geografia

Pergunta: {input}

Responda apenas com o nome da categoria.
""")

router_chain = router_prompt | chat | StrOutputParser()

In [4]:
def route(info):
    destino = info["topic"]
    print(f"[ROUTER] Escolheu: {destino}")

    if destino == "quimica":
        print("[CHAIN] Usando cadeia de QUÍMICA")
        return chain_quimica
    else:
        print("[CHAIN] Usando cadeia de GEO")
        return chain_geo

In [5]:
full_chain = (
    {
        "topic": router_chain,
        "input": lambda x: x["input"]
    }
    | RunnableLambda(route)
    | (lambda chain: chain)
)

In [6]:
full_chain.invoke({"input": "Onde fica o Brasil?"})


[ROUTER] Escolheu: geografia
[CHAIN] Usando cadeia de GEO


AIMessage(content='O Brasil está localizado na América do Sul. Ele ocupa a maior parte da porção oriental e central do continente, fazendo fronteira com quase todos os países sul‑americanos: a norte com a Guiana Francesa, Suriname, Guiana, Venezuela e Colômbia; a noroeste com a Colômbia; a oeste com a Bolívia e o Peru; a sudoeste com a Argentina, Paraguai e Uruguai. A costa brasileira banha o Oceano Atlântico, estendendo‑se de aproximadamente 5°\u202fN (na região do Amapá) até 34°\u202fS (no Rio Grande do Sul). Em termos de coordenadas geográficas, o centro aproximado do território brasileiro está em torno de 10°\u202fS de latitude e 55°\u202fW de longitude.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 194, 'prompt_tokens': 84, 'total_tokens': 278, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 15, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {